<a href="https://colab.research.google.com/github/Jeshurun-B/EMA-optimizer-pipeline-v2/blob/main/coolab_notebooks/week3_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📅 Week 3 — Feature Engineering + Rigorous Model Evaluation
**EMA Crossover ML Project | 10-Week Curriculum**
**Date:** June 21, 2026

---

### What This Week Is About

Week 2 proved that Logistic Regression can separate good signals from bad ones — you built a full classification pipeline, tuned thresholds using Youden's J, explored regularisation, and constructed logical ensembles of specialist classifiers. But your validation was simple: an 80/20 chronological split. This week you fix that permanently by building the **walk-forward cross-validation framework** that every future model will use. You will also engineer new features — interactions, ratios, domain-specific constructs, and lag features — that give non-linear models the raw material they need to beat logistic regression in Weeks 4–7. This is the infrastructure week: the engineering and evaluation machinery you build here will compound in value every week that follows.

**By end of week you will have:**
- A reusable `walk_forward_cv()` function that every future model uses (no exceptions)
- 15+ interaction and ratio features engineered from your existing 35
- 10+ domain-specific features derived from the analytics table
- A justified final feature set of 20–30 columns with selection rationale
- A retrained Logistic Regression baseline on the engineered feature set (your new benchmark)


## Section 0 — Environment Setup

Run this first every session. Added `statsmodels` this week for VIF calculations.

In [ ]:
# Install required libraries (added statsmodels for VIF)
!pip install -q supabase scikit-learn statsmodels

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from supabase import create_client
from google.colab import userdata

# sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    precision_recall_curve, average_precision_score
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFE

# statsmodels for VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('Environment ready ✓')


## Section 1 — Connect to Both Databases

Same dual-connection pattern. Copy-paste from Week 2. Do not change it.

In [ ]:
# ============================================================
# CONNECT TO MAIN DATABASE
# ============================================================
try:
    main_client = create_client(userdata.get('SUPABASE_URL'), userdata.get('SUPABASE_KEY'))
    print('Main database connected ✓')
except Exception as e:
    print(f'Main connection failed: {e}')


In [ ]:
# ============================================================
# CONNECT TO ANALYTICS DATABASE
# ============================================================
try:
    analytics_client = create_client(
        userdata.get('ANALYTICS_SUPABASE_URL'),
        userdata.get('ANALYTICS_SUPABASE_KEY')
    )
    print('Analytics database connected ✓')
except Exception as e:
    print(f'Analytics connection failed: {e}')


## Section 2 — Fetch Data and Merge

Same merge pattern from Weeks 1–2. Inner join on `(symbol, checked_at_utc)`.

In [ ]:
# ============================================================
# FETCH: Main signals table
# ============================================================
response_main = main_client.table('signals').select('*').eq('status', 'analyzed').execute()
df_main = pd.DataFrame(response_main.data)

for col in ['checked_at_utc', 'time_of_max_price', 'time_of_min_price']:
    if col in df_main.columns:
        df_main[col] = pd.to_datetime(df_main[col], utc=True, errors='coerce')

df_main = df_main.sort_values('checked_at_utc').reset_index(drop=True)
print(f'Main signals: {len(df_main):,} rows | {df_main.shape[1]} columns')
print(f'Range: {df_main["checked_at_utc"].min().date()} to {df_main["checked_at_utc"].max().date()}')
print(f'Signal split: {df_main["signal"].value_counts().to_dict()}')


In [ ]:
# ============================================================
# FETCH: Analytics table
# ============================================================
response_analytics = analytics_client.table('crossover_analytics').select('*').execute()
df_analytics = pd.DataFrame(response_analytics.data)

for col in ['crossover_utc', 'next_crossover_utc', 'optimal_entry_utc']:
    if col in df_analytics.columns:
        df_analytics[col] = pd.to_datetime(df_analytics[col], utc=True, errors='coerce')

df_analytics = df_analytics.sort_values('crossover_utc').reset_index(drop=True)
print(f'Analytics: {len(df_analytics):,} rows | {df_analytics.shape[1]} columns')


In [ ]:
# ============================================================
# MERGE: inner join on (symbol, checked_at_utc)
# ============================================================
df_analytics_renamed = df_analytics.rename(columns={'crossover_utc': 'checked_at_utc'})
df = pd.merge(df_main, df_analytics_renamed, how='inner', on=['checked_at_utc', 'symbol'])

main_lost      = len(df_main) - len(df)
analytics_lost = len(df_analytics) - len(df)

print(f'Merged dataset: {len(df):,} rows | {df.shape[1]} columns')
print(f'Lost from main  (no analytics match): {main_lost:,} ({main_lost/len(df_main)*100:.1f}%)')
print(f'Lost from analytics (no main match): {analytics_lost:,}')
print()
print('Records per symbol:')
print(df['symbol'].value_counts())


## Section 3 — Apply Signal Quality Columns

Locked scoring module from Week 1. Applied every week after merge. Do not modify.

In [ ]:
# ============================================================
# SIGNAL QUALITY SCORING MODULE (Week 1 final version — DO NOT MODIFY)
# ============================================================

def get_time_decay_score(candles):
    if candles <= 20:
        return 1.0
    elif candles <= 100:
        t = (candles - 20) / 80
        return np.exp(-3.5 * t)
    elif candles <= 400:
        base = np.exp(-3.5)
        t = (candles - 100) / 300
        return base * np.exp(-4 * t)
    else:
        return 0.0

def time_cutoff(candles):
    if candles <= 20:   return 1.0
    elif candles <= 40: return 0.5
    else:               return 0

def calculate_four_target_scores(row):
    is_long = str(row['signal_x']).upper() == 'LONG'
    if is_long:
        p, r = float(row['max_move_up_pct']), float(row['max_move_down_pct'])
        candles_to_peak_profit = int(row['candles_to_max_price'])
        candles_to_max_pain    = int(row['candles_to_min_price'])
        is_btc  = bool(row['btc_trend_bias'])
        is_1d   = bool(row['htf_1d_bias'])
        is_4h   = bool(row['htf_4h_bias'])
    else:
        p, r = float(row['max_move_down_pct']), float(row['max_move_up_pct'])
        candles_to_peak_profit = int(row['candles_to_min_price'])
        candles_to_max_pain    = int(row['candles_to_max_price'])
        is_btc  = not bool(row['btc_trend_bias'])
        is_1d   = not bool(row['htf_1d_bias'])
        is_4h   = not bool(row['htf_4h_bias'])

    r_safe      = 0.01 if r <= 0 else r
    current_rr  = p / r_safe
    t_factor    = get_time_decay_score(candles_to_peak_profit)
    base        = round(min(5.0, current_rr) + 5.0 * t_factor, 2)

    t1 = base

    if candles_to_max_pain < candles_to_peak_profit and r >= p:
        t2 = round(base * 0.2, 2)
    elif r < 0.25:
        t2 = round(base * 0.1, 2)
    else:
        t2 = base

    pen = base
    if 'volume_ratio' in row and float(row['volume_ratio']) < 1.0: pen -= 0.8
    if not is_btc:  pen -= 0.5
    if not is_1d:   pen -= 1.0
    if not is_4h:   pen -= 1.2
    if candles_to_max_pain < candles_to_peak_profit and r > 0.75 * p: pen -= 0.8
    t3 = round(max(0, pen), 2)

    if r < 0.25 or (candles_to_max_pain < candles_to_peak_profit and r >= p):
        t4 = 0.0
    else:
        if current_rr >= 4:              rr_p = 5.0
        elif current_rr >= 1.5:          rr_p = current_rr
        elif 1.0 <= current_rr < 1.5:   rr_p = (current_rr**7) / (1.5**6)
        else:                             rr_p = 0.0
        t_s = time_cutoff(candles_to_peak_profit)
        t4  = round(5.0 * t_s + rr_p, 2) if rr_p > 0 else 0.0

    return t1, t2, t3, t4


def optimal_entry_candles(row):
    """Returns 15-min candles from crossover to optimal entry. 0 = immediate entry."""
    try:
        if pd.isnull(row['optimal_entry_utc']) or pd.isnull(row['checked_at_utc']):
            return 0.0
        diff = row['optimal_entry_utc'] - row['checked_at_utc']
        return float(diff.total_seconds() / 60 / 15)
    except Exception:
        return 0.0


# Apply
target_cols = ['T1_Pure_Continuous', 'T2_Soft_Floor', 'T3_Assumption_Penalty', 'T4_Control_Punished']
df[target_cols] = df.apply(lambda r: pd.Series(calculate_four_target_scores(r)), axis=1)
df['target_special'] = df[target_cols[:3]].min(axis=1)
df['Optimum_entry']  = df.apply(optimal_entry_candles, axis=1)

print('All quality columns applied ✓')
print('\n--- % scoring above 5.0 ---')
for col in target_cols + ['target_special']:
    print(f'  {col:<28} -> {(df[col] > 5).mean()*100:.1f}%')
print(f'\n  Optimum_entry median: {df["Optimum_entry"].median():.1f} candles | mean: {df["Optimum_entry"].mean():.1f}')


## Section 4 — Carry Forward From Week 2

Re-establish your binary target definition and Week 2 feature sets before building anything new. Your Week 2 chosen target is the reference point for measuring whether new features help.

In [ ]:
# ============================================================
# FEATURE SET DEFINITIONS (Week 2 final versions)
# ============================================================

FEATURES_ALL = [
    'ema_fast_ltf', 'ema_slow_ltf', 'ema_fast_slope', 'ema_slow_slope',
    'ema_separation', 'price_above_both_emas', 'crossover_candle_strength',
    'adx_ltf', 'adx_slope', 'adx_4h', 'macd_histogram_ltf', 'macd_histogram_4h',
    'htf_4h_bias', 'htf_1d_bias', 'ema_separation_4h', 'rsi_4h',
    'rsi_ltf', 'roc_ltf', 'atr_ltf', 'atr_pct', 'bb_width_ltf', 'price_to_atr',
    'volume_ratio', 'volume_trend', 'crossover_volume_ratio',
    'fear_greed_index', 'btc_trend_bias', 'hour_of_day', 'day_of_week',
    'swing_high', 'swing_low', 'atr_stop_distance', 'signal_gap_hours'
]

FEATURES_LONG_MI = [
    'volume_ratio', 'btc_trend_bias', 'rsi_4h', 'htf_4h_bias',
    'ema_separation_4h', 'ema_fast_slope', 'ema_slow_slope', 'price_to_atr',
    'atr_pct', 'bb_width_ltf', 'volume_trend', 'htf_1d_bias',
    'atr_ltf', 'macd_histogram_ltf', 'rsi_ltf'
]

FEATURES_SHORT_MI = [
    'rsi_4h', 'htf_4h_bias', 'ema_separation_4h', 'btc_trend_bias',
    'ema_fast_slope', 'ema_slow_slope', 'atr_pct', 'htf_1d_bias',
    'price_to_atr', 'crossover_volume_ratio', 'volume_ratio', 'roc_ltf',
    'bb_width_ltf', 'atr_stop_distance', 'atr_ltf'
]

# ============================================================
# BINARY TARGET: Carry forward your Week 2 chosen definition
# ============================================================
# TODO: Re-apply whichever binary target definition you settled on in Week 2.
#       Paste it here so the baseline comparison cell in Section 10 can use it.
#       Label it: df['binary_target'] = ...

THRESHOLD_T3  = 5.3   # TODO: update to your Week 2 final threshold
THRESHOLD_MFE = 0.5   # TODO: update to your Week 2 final mfe threshold

# TODO: reconstruct df['binary_target'] using the same logic as Week 2

print(f'FEATURES_ALL:      {len(FEATURES_ALL)} features')
print(f'FEATURES_LONG_MI:  {len(FEATURES_LONG_MI)} features')
print(f'FEATURES_SHORT_MI: {len(FEATURES_SHORT_MI)} features')
# TODO: print class balance for binary_target (LONG and SHORT separately)


## Section 5 — Walk-Forward CV Framework (Monday)

### Why This Section Is the Most Important Thing You Build This Week

Your Week 2 evaluation used a single 80/20 train/test split. That gives you one data point — one AUC, one F1. You don't know whether that number is real or lucky. What you need is a distribution of scores across multiple time periods.

**Why standard k-fold is wrong for time series:**
Standard k-fold randomly shuffles the data before splitting into folds. For a time series, this means fold N might train on January candles and test on December candles, while *also* including December data in another training fold. The model learns from the future during training — this is **look-ahead leakage**. Your CV score becomes optimistically inflated, and you deploy a model that looked great in testing but has never actually seen unseen data.

`TimeSeriesSplit` enforces a strict rule: **the test fold is always chronologically after the training fold**. Fold 1 trains on the first 20% and tests on the next 20%. Fold 2 trains on the first 40% and tests on the next 20%. And so on. You never look forward.

**Gap parameter:** Consider adding a gap between the end of training and the start of testing. If a crossover signal at 23:45 influences one at 00:15 the next day, a gap of 2–3 periods prevents the boundary from leaking.

**Your job here:** Build a reusable `walk_forward_cv()` function that takes any model, X, y and returns a dictionary of mean ± std scores across all folds. Every model from Week 3 onward MUST be evaluated through this function — not a one-shot split.


In [ ]:
# ============================================================
# WALK-FORWARD CV FRAMEWORK — REUSABLE FROM THIS WEEK ONWARD
# ============================================================
# This function must be called for every model in Weeks 3-10.
# Do not evaluate models any other way.

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

def walk_forward_cv(model, X, y, n_splits=5, gap=0):
    """
    Walk-forward time series cross-validation.

    Parameters
    ----------
    model   : sklearn-compatible estimator (must have fit, predict, predict_proba)
    X       : np.ndarray, feature matrix (already scaled if needed)
    y       : np.ndarray, binary target
    n_splits: int, number of CV folds (default 5)
    gap     : int, number of samples to skip between train and test
              (set > 0 if adjacent rows can leak information)

    Returns
    -------
    dict: {metric: (mean, std)} for accuracy, precision, recall, f1, auc
    """
    # TODO: Instantiate TimeSeriesSplit with n_splits and gap parameters
    tscv = None  # TODO: TimeSeriesSplit(n_splits=n_splits, gap=gap)

    scores = {
        'accuracy':  [],
        'precision': [],
        'recall':    [],
        'f1':        [],
        'auc':       [],
    }

    # TODO: Loop over tscv.split(X)
    #       For each (train_idx, test_idx):
    #         - Slice X_tr, X_te, y_tr, y_te
    #         - model.fit(X_tr, y_tr)
    #         - y_pred = model.predict(X_te)
    #         - y_prob = model.predict_proba(X_te)[:, 1]
    #         - Guard against single-class folds (skip with continue if needed)
    #         - Append each metric to the scores dict

    # TODO: return {k: (np.mean(v), np.std(v)) for k, v in scores.items()}
    pass


def print_cv_results(results, model_name='Model'):
    """Pretty-print the output of walk_forward_cv."""
    print(f'\n=== Walk-Forward CV: {model_name} ===')
    for metric, (mean, std) in results.items():
        print(f'  {metric:<12}: {mean:.4f} ± {std:.4f}')
    print()


print('walk_forward_cv() defined ✓')
print('print_cv_results() defined ✓')


In [ ]:
# ============================================================
# VALIDATE THE CV FRAMEWORK — Does it behave as expected?
# ============================================================
# Before using walk_forward_cv on your real model, sanity-check it.
#
# What you are checking:
#   1. Fold sizes grow monotonically (each training fold is bigger than the last)
#   2. Test folds never overlap with training folds
#   3. The gap, if any, is respected
#   4. No fold contains a single class (which would make AUC undefined)

# TODO: Prepare a minimal dataset for this test
#   df_long = df[df['signal_x'] == 'LONG'].sort_values('checked_at_utc').reset_index(drop=True)
#   Use FEATURES_LONG_MI and your binary_target column
#   Apply StandardScaler (fit on training portion only — but for this validation,
#   scaling the whole thing to inspect fold shapes is fine)

# TODO: Loop over TimeSeriesSplit(n_splits=5).split(X):
#   Print for each fold:
#     f'Fold {i}: train rows {len(train_idx)}, test rows {len(test_idx)}'
#     f'  Train dates: {df_long.iloc[train_idx]["checked_at_utc"].min().date()} to ...'
#     f'  Test  dates: {df_long.iloc[test_idx]["checked_at_utc"].min().date()} to ...'

# TODO: After printing, confirm in a comment:
#   - Do test start dates advance chronologically fold-by-fold?
#   - Is there any date overlap between train and test in any fold?
#   - Are folds reasonably sized for training a logistic model?


In [ ]:
# ============================================================
# BASELINE CV SCORE — Week 2 Logistic Regression through the CV framework
# ============================================================
# Goal: establish what your WEEK 2 best model scores under walk-forward CV.
# This is the number every engineered feature set must beat.

# TODO: Prepare X (FEATURES_LONG_MI) and y (binary_target) for LONG signals
#   Sort by checked_at_utc. Do NOT shuffle.

# TODO: Scale X with StandardScaler.
#       Important: for a proper CV you should scale INSIDE each fold.
#       For this baseline cell, scaling globally is acceptable — you will
#       fix this properly when you build the full pipeline in Section 10.

# TODO: Instantiate your Week 2 best Logistic Regression
#   lr_baseline = LogisticRegression(C=???, class_weight='balanced',
#                                     max_iter=1000, solver='lbfgs')
#   (fill in your best C from Week 2)

# TODO: cv_results_baseline = walk_forward_cv(lr_baseline, X_sc, y, n_splits=5)
# TODO: print_cv_results(cv_results_baseline, 'LR Baseline (Week 2 features)')

# ── IMPORTANT REFLECTION ──────────────────────────────────────────────────────
# TODO: Compare this CV AUC to your Week 2 single held-out test AUC.
#       Are they close? If the CV AUC is lower, the single split was optimistic.
#       Write your finding in a comment below.


## Section 6 — Feature Engineering: Interaction + Ratio Features (Tuesday)

### What You Are Doing and Why

Logistic Regression is a **linear** model — it can only draw straight-line decision boundaries. If a signal is good when BOTH ADX is high AND volume is elevated (but not when only one is true), logistic regression cannot capture that without help. You give it that help by explicitly computing the product `adx_ltf × volume_ratio` as a new column.

**Types of features you will engineer here:**

- **Interaction features (products):** Capture "both conditions at once" patterns that a linear boundary cannot. `adx_ltf × volume_ratio` spikes only when trend strength AND volume confirm each other.
- **Ratio features:** Normalise one quantity relative to another. `ema_fast_ltf / ema_slow_ltf` removes the price-level dependence from the raw EMA values, leaving only their relative spread — which is what matters for crossover strength.
- **Boolean / regime flags:** Discretise continuous features into hard signals that are meaningful in trading context. `adx_ltf > 25` separates trending from ranging regimes. RSI extremes separate momentum from exhaustion.

**Naming convention:** Prefix all engineered features with `FE_` so you can always tell them from raw features (e.g. `FE_adx_x_volume`, `FE_rsi_ema_ratio`). This makes feature selection bookkeeping much easier.

**Do not look at the target while engineering.** Decide what to compute based on market logic — "these two things should interact" — not by checking which combinations correlate with your label. That is target leakage via feature construction.


In [ ]:
# ============================================================
# INTERACTION FEATURES (products of two raw features)
# ============================================================
# Each of these captures a "both conditions active" scenario.

# TODO: FE_adx_x_volume      = df['adx_ltf'] * df['volume_ratio']
# TODO: FE_rsi_x_htf4h       = df['rsi_ltf'] * df['htf_4h_bias']
# TODO: FE_macd_x_volume     = df['macd_histogram_ltf'] * df['volume_ratio']
# TODO: FE_adx_x_htf1d       = df['adx_ltf'] * df['htf_1d_bias']
# TODO: FE_ema_sep_x_adx     = df['ema_separation'] * df['adx_ltf']
# TODO: FE_rsi4h_x_htf1d     = df['rsi_4h'] * df['htf_1d_bias']
# TODO: FE_volume_x_candle   = df['volume_ratio'] * df['crossover_candle_strength']

# Add as many more as you think are meaningful. Explain your reasoning in a comment.

# TODO: After computing, print a quick sanity check:
#   - Shape of df
#   - First 5 rows of just the FE_ columns
#   - NaN count per new column (any unexpected nulls?)

print('Interaction features computed ✓')


In [ ]:
# ============================================================
# RATIO FEATURES (one quantity relative to another)
# ============================================================
# Guard against division by zero on every ratio.

# TODO: FE_ema_ratio         = df['ema_fast_ltf'] / df['ema_slow_ltf'].replace(0, np.nan)
# TODO: FE_price_to_bb       = df['atr_pct'] / df['bb_width_ltf'].replace(0, np.nan)
# TODO: FE_adx_4h_ratio      = df['adx_ltf'] / df['adx_4h'].replace(0, np.nan)
# TODO: FE_mfe_mae_ratio     = df['mfe_percent'] / df['mae_percent'].abs().replace(0, np.nan)
#       (this is the realised R:R from the analytics table — powerful signal)
# TODO: FE_slope_ratio       = df['ema_fast_slope'] / df['ema_slow_slope'].replace(0, np.nan)

# TODO: Fill any resulting NaN / inf values — decide: ffill, 0, or median?
#       Document your choice and why.

# TODO: NaN count per new ratio column

print('Ratio features computed ✓')


In [ ]:
# ============================================================
# BOOLEAN / REGIME FLAGS
# ============================================================
# Binary features that capture market-structure thresholds.

# TODO: FE_adx_trending      = (df['adx_ltf'] > 25).astype(int)
# TODO: FE_adx_4h_trending   = (df['adx_4h'] > 25).astype(int)
# TODO: FE_rsi_overbought    = (df['rsi_ltf'] > 70).astype(int)
# TODO: FE_rsi_oversold      = (df['rsi_ltf'] < 30).astype(int)
# TODO: FE_rsi_4h_bull       = (df['rsi_4h'] > 55).astype(int)
# TODO: FE_full_htf_align    = ((df['htf_4h_bias'] == 1) & (df['htf_1d_bias'] == 1)).astype(int)
#       (for LONG — need separate logic for SHORT)
# TODO: FE_high_volume       = (df['volume_ratio'] > 1.5).astype(int)
# TODO: FE_btc_volume_align  = ((df['btc_trend_bias'] == 1) & (df['volume_ratio'] > 1.0)).astype(int)

# TODO: For each boolean flag, print the % of rows where it is 1.
#       Are any flags near 0% or 100%? If so, they are useless (near-zero variance).

print('Boolean regime flags computed ✓')


## Section 7 — Feature Engineering: Domain-Specific Features (Wednesday)

### What You Are Doing and Why

The previous section computed statistical combinations of existing raw features. This section derives new signals from **market structure and trading domain knowledge** — things a professional trader would look at but that don't exist as columns in your database yet.

These features are more powerful because they encode *context*, not just combinations. A signal arriving after a losing trade in the same direction is structurally different from a first signal in a quiet period. A signal during the London-NY overlap has more liquidity and momentum behind it than a 3am Asian session signal.

**Key features to engineer this session:**

- **Lag features:** What was the quality and direction of the previous signal for the same symbol? Whipsaw clusters look different from isolated signals.
- **Market session flags:** Asia (23:00–08:00 UTC), London (07:00–16:00 UTC), New York (13:00–22:00 UTC), London-NY overlap (13:00–16:00 UTC). Session context affects volatility and follow-through.
- **Regime features:** ADX bucketed into regimes (ranging / trending / strong trend). RSI zone (oversold / neutral / overbought). These capture non-linear market states.
- **Realised R:R from analytics:** `mfe_percent / abs(mae_percent)` is the actual achieved risk-reward from the analytics table. Use it as a feature input for the MODEL (not as a target label — it comes from the same record as the target, so you must be careful about whether it constitutes leakage).
- **Signal gap quality:** How long since the last crossover signal? A signal appearing 2 hours after the last one is structurally different from one appearing 3 days later.

**Leakage warning on analytics features:** `mfe_percent`, `mae_percent`, `pnl_percent`, `trade_duration` are post-signal outcomes. They are perfectly valid as features if the previous row's outcome is used to describe the *current* row's context (lag). Using the *same* row's mfe as a feature to predict whether it is a good trade is circular — the model would be predicting the outcome FROM the outcome. Use lag-1 analytics values only.


In [ ]:
# ============================================================
# LAG FEATURES (previous signal's context)
# ============================================================
# Sort by (symbol, checked_at_utc) before shifting — critical.
# Shift(1) within each symbol group gives the previous row's value.

df = df.sort_values(['symbol', 'checked_at_utc']).reset_index(drop=True)

# TODO: FE_prev_signal_dir   = df.groupby('symbol')['signal_x'].shift(1)
#       Encode as: LONG=1, SHORT=-1, NaN=0
#       (previous trade direction for the same coin)

# TODO: FE_prev_t3_score     = df.groupby('symbol')['T3_Assumption_Penalty'].shift(1)
#       (was the previous signal high or low quality?)

# TODO: FE_prev_mfe          = df.groupby('symbol')['mfe_percent'].shift(1)
#       (did the previous trade move in the right direction at all?)

# TODO: FE_same_dir_streak   = ...
#       Count how many consecutive same-direction signals preceded this one.
#       Hint: group by symbol, then use a rolling or cumsum trick on signal direction.
#       This is the "whipsaw cluster" detector.

# TODO: FE_signal_gap_candles = df['signal_gap_hours'] * 4
#       Convert signal_gap_hours to 15-min candle count.
#       (signal_gap_hours already exists — this just changes the unit)

# TODO: After computing, fillna(0) for lag features (first row per symbol has no lag)
# TODO: Print NaN counts before and after fill

print('Lag features computed ✓')


In [ ]:
# ============================================================
# MARKET SESSION FEATURES
# ============================================================
# Crypto trades 24/7 but liquidity is not uniform.
# London-NY overlap (13:00-16:00 UTC) has the highest volume.
# Extract hour from checked_at_utc (UTC) for session flags.

# TODO: Ensure checked_at_utc is tz-aware (it should be from Section 2)
# TODO: FE_hour_utc = df['checked_at_utc'].dt.hour  (if not already 'hour_of_day')

# Session definitions (UTC hours):
# Asia session:          23, 0, 1, 2, 3, 4, 5, 6, 7
# London session:        7, 8, 9, 10, 11, 12, 13, 14, 15
# New York session:      13, 14, 15, 16, 17, 18, 19, 20, 21
# London-NY overlap:     13, 14, 15

# TODO: FE_session_asia      = df['hour_of_day'].isin([23, 0, 1, 2, 3, 4, 5, 6, 7]).astype(int)
# TODO: FE_session_london    = df['hour_of_day'].isin([7, 8, 9, 10, 11, 12, 13, 14, 15]).astype(int)
# TODO: FE_session_ny        = df['hour_of_day'].isin([13, 14, 15, 16, 17, 18, 19, 20, 21]).astype(int)
# TODO: FE_session_overlap   = df['hour_of_day'].isin([13, 14, 15]).astype(int)
# TODO: FE_weekend           = (df['day_of_week'] >= 5).astype(int)
#       (day_of_week: Monday=0 … Sunday=6 in pandas)

# TODO: Print percentage of signals in each session. Are they balanced?
#       Does one session dominate? What might that imply for model generalisation?

print('Market session features computed ✓')


In [ ]:
# ============================================================
# REGIME + DOMAIN FEATURES
# ============================================================
# These encode market structure states that matter for signal quality.

# TODO: FE_adx_regime = pd.cut(df['adx_ltf'],
#                              bins=[0, 20, 35, 100],
#                              labels=[0, 1, 2])   # 0=ranging, 1=trending, 2=strong trend
#       Cast to int. Print value counts.

# TODO: FE_rsi_zone = pd.cut(df['rsi_ltf'],
#                            bins=[0, 30, 45, 55, 70, 100],
#                            labels=[0, 1, 2, 3, 4])  # 0=oversold … 4=overbought
#       Cast to int. Print value counts.

# TODO: FE_mfe_mae_ratio_lag = df.groupby('symbol')['mfe_percent'].shift(1) / #                              df.groupby('symbol')['mae_percent'].abs().shift(1).replace(0, np.nan)
#       This is the LAGGED realised R:R (safe from leakage — it's the previous trade's outcome)
#       Fill NaN with median.

# TODO: FE_optimum_entry_bin = pd.cut(df['Optimum_entry'],
#                                     bins=[-1, 0.5, 5, 20, 1000],
#                                     labels=[0, 1, 2, 3])
#       0=immediate, 1=short wait (1-5 candles), 2=medium wait, 3=long wait
#       Entry timing cluster — quick entries may have different quality profiles

# TODO: Print a crosstab of FE_adx_regime vs binary_target to see if the regime
#       discriminates signal quality (it should).

print('Regime + domain features computed ✓')


In [ ]:
# ============================================================
# COLLECT ALL ENGINEERED FEATURE NAMES
# ============================================================
# Keep a clean master list so you don't have to search through df.columns later.

FE_FEATURES = [col for col in df.columns if col.startswith('FE_')]

print(f'Total engineered features created: {len(FE_FEATURES)}')
print()
for f in FE_FEATURES:
    print(f'  {f}')

# TODO: Sanity check — any remaining NaN in FE features?
nan_report = df[FE_FEATURES].isnull().sum()
print()
print('NaN counts per FE feature:')
print(nan_report[nan_report > 0])

# Combined feature set (original MI features + all engineered features)
FEATURES_COMBINED_LONG  = FEATURES_LONG_MI  + FE_FEATURES
FEATURES_COMBINED_SHORT = FEATURES_SHORT_MI + FE_FEATURES

print(f'\nCombined LONG feature set:  {len(FEATURES_COMBINED_LONG)} features')
print(f'Combined SHORT feature set: {len(FEATURES_COMBINED_SHORT)} features')


## Section 8 — Feature Selection (Thursday)

### Why You Need This

You now have 15 MI features + potentially 20+ engineered features. More features is not always better:

- **Multicollinearity:** If `adx_ltf` and `FE_adx_trending` are highly correlated, they both carry the same information. Including both wastes a degree of freedom and inflates the model's uncertainty about individual coefficients.
- **Noise features:** Some engineered features won't actually be useful. Including them adds noise that can lower generalisation.
- **Computational cost:** As models get more complex (trees, boosting), the feature space affects training time quadratically.

**Three tools you'll use:**

1. **Correlation filtering** — Fast. Remove one of any pair with Pearson `|r| > 0.90`. Which one to drop? Keep the one that is more theoretically meaningful.
2. **VIF (Variance Inflation Factor)** — More rigorous than correlation. VIF > 10 for a feature means it can be almost perfectly predicted from the other features — it is redundant. VIF > 5 is a warning. Iteratively remove the highest VIF until all are below 5.
3. **Permutation importance** — Train a quick model, then randomly shuffle one feature at a time and measure how much performance drops. A feature with near-zero permutation importance doesn't help the model, even if it looks correlated with the target on its own.

**Your goal:** A final feature set of 20–30 features with a documented justification for each inclusion and exclusion.


In [ ]:
# ============================================================
# STEP 1: CORRELATION FILTERING
# ============================================================
# Remove one of any pair of features with |Pearson r| > 0.90

# TODO: Select your combined feature set columns from df (for LONG signals only)
#   df_long = df[df['signal_x'] == 'LONG'].copy()
#   corr_matrix = df_long[FEATURES_COMBINED_LONG].corr().abs()

# TODO: Extract upper triangle of the correlation matrix (to avoid double-counting pairs)
#   upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# TODO: Find all feature pairs where correlation > 0.90
#   high_corr_pairs = [(col, row) for col in upper.columns
#                      for row in upper.index if upper.loc[row, col] > 0.90]

# TODO: For each pair, print both features and their correlation value.
#       Decide which to drop. Prefer dropping engineered features over raw features
#       IF the raw feature is already in FEATURES_LONG_MI (it's been validated).

# TODO: Build a list: FEATURES_TO_DROP_CORR = [...]
# TODO: FEATURES_AFTER_CORR = [f for f in FEATURES_COMBINED_LONG if f not in FEATURES_TO_DROP_CORR]

print('Correlation filtering complete ✓')
# TODO: print(f'{len(FEATURES_TO_DROP_CORR)} features dropped, {len(FEATURES_AFTER_CORR)} remain')


In [ ]:
# ============================================================
# STEP 2: VIF (VARIANCE INFLATION FACTOR)
# ============================================================
# VIF > 10 = severe multicollinearity → drop
# VIF > 5  = worth investigating

# TODO: Drop rows with any NaN in FEATURES_AFTER_CORR (VIF cannot handle NaN)
#   df_vif = df_long[FEATURES_AFTER_CORR].dropna()

# TODO: Compute VIF for each feature:
#   from statsmodels.stats.outliers_influence import variance_inflation_factor
#   vif_data = pd.DataFrame()
#   vif_data['feature'] = FEATURES_AFTER_CORR
#   vif_data['VIF'] = [variance_inflation_factor(df_vif.values, i)
#                      for i in range(len(FEATURES_AFTER_CORR))]
#   vif_data = vif_data.sort_values('VIF', ascending=False)

# TODO: Print vif_data. Highlight any VIF > 5.

# TODO: Iterative removal:
#   Drop the feature with the highest VIF (if VIF > 5).
#   Recompute VIF for the remaining set.
#   Repeat until all VIF < 5.
#   Keep a log of what you dropped and its VIF at time of removal.

# NOTE: If statsmodels VIF calculation is slow on your data, sample 2,000 rows first.

print('VIF filtering complete ✓')


In [ ]:
# ============================================================
# STEP 3: PERMUTATION IMPORTANCE
# ============================================================
# Train a quick Logistic Regression, then measure how much performance
# drops when you randomly shuffle each feature. Features with near-zero
# permutation importance don't help the model — drop them.

# TODO: Prepare X and y from FEATURES_AFTER_VIF (your VIF-filtered set)
#   df_long_clean = df_long[FEATURES_AFTER_VIF + ['binary_target', 'checked_at_utc']].dropna()
#   df_long_clean = df_long_clean.sort_values('checked_at_utc').reset_index(drop=True)
#   split_idx = int(len(df_long_clean) * 0.8)
#   train_set = df_long_clean.iloc[:split_idx]
#   test_set  = df_long_clean.iloc[split_idx:]

# TODO: Scale with StandardScaler (fit on train only)
# TODO: Train LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000)

# TODO: from sklearn.inspection import permutation_importance
#   perm_result = permutation_importance(
#       lr_model, X_test_sc, y_test,
#       n_repeats=20, scoring='roc_auc', random_state=42
#   )

# TODO: Build DataFrame of (feature, mean_importance, std_importance)
#       Sort descending by mean_importance

# TODO: Horizontal bar chart of permutation importance (top 30 features)
#       Colour bars: green if mean > 0, red if mean <= 0

# TODO: Features to drop: those with mean permutation importance <= 0
#       These features actively HURT the model when shuffled → they add noise

# TODO: FEATURES_FINAL_LONG = [features where perm importance > 0]
#       Print count: f'{len(FEATURES_FINAL_LONG)} features in final LONG set'

print('Permutation importance filtering complete ✓')


In [ ]:
# ============================================================
# FINAL FEATURE SET: DOCUMENT YOUR SELECTIONS
# ============================================================
# Fill in your justified final feature set below.
# This must include every feature carried forward AND a one-line rationale.

# TODO: FEATURES_FINAL_LONG = [
#     'feature_name',   # rationale: why this feature survived all three filters
#     ...
# ]

# TODO: FEATURES_FINAL_SHORT = [
#     # Repeat the same exercise for SHORT signals
#     # You can reuse the LONG set as a starting point but run the filters
#     # on the SHORT subset separately — features may perform differently
#     ...
# ]

# TODO: Print summary table:
#   Original FEATURES_ALL:    33 features
#   After MI selection (W2):  15 features
#   After FE additions:       X features
#   After correlation filter: X features
#   After VIF filter:         X features
#   After permutation filter: X features  ← FINAL SET

print('Final feature sets defined ✓')


## Section 9 — Model Re-evaluation with Engineered Features (Friday)

### What You Are Doing

You now have a final feature set. Retrain Logistic Regression on it through the walk-forward CV framework. Compare rigorously to your Week 2 baseline. The comparison table at the bottom of this section is your **Week 3 Honest Record** — fill in real numbers.

### Proper Scaling Inside CV Folds

In the validation cell (Section 5), you scaled globally before CV. That is technically wrong — it leaks statistics from test folds into the scaler fitted on training data. In production pipelines you scale *inside* each fold. The code below shows the correct pattern.

For logistic regression the effect is small. For more complex models later, this matters more. Build the habit now.


In [ ]:
# ============================================================
# LOGISTIC REGRESSION: FINAL FEATURE SET vs WEEK 2 BASELINE
# ============================================================
# Correct CV pipeline: scaler is fit inside each fold.

def cv_with_scaling(model, df_subset, feature_cols, target_col, n_splits=5):
    """
    Walk-forward CV where StandardScaler is fit INSIDE each training fold.
    This is the correct pattern — use it from here onward.

    Parameters
    ----------
    model        : sklearn estimator (unfitted)
    df_subset    : DataFrame sorted by checked_at_utc
    feature_cols : list of feature column names
    target_col   : str, name of binary target column
    n_splits     : int

    Returns
    -------
    dict: {metric: (mean, std)}
    """
    # TODO: Drop rows with NaN in feature_cols or target_col
    # TODO: Sort by checked_at_utc
    # TODO: Extract X (np.ndarray) and y (np.ndarray)

    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'auc': []}

    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        # TODO: Fit scaler on X_tr only, transform both
        # TODO: Guard against single-class folds
        # TODO: model.fit(X_tr_sc, y_tr)
        # TODO: Compute and append all metrics

    # TODO: return {k: (np.mean(v), np.std(v)) for k, v in scores.items()}
    pass


print('cv_with_scaling() defined ✓')


In [ ]:
# ============================================================
# RUN: Comparison across 4 feature sets
# ============================================================
# LONG signals — compare 4 configurations through walk-forward CV

df_long = df[df['signal_x'] == 'LONG'].sort_values('checked_at_utc').reset_index(drop=True)

lr_model = LogisticRegression(
    # TODO: fill in your best C from Week 2
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)

# TODO: Run cv_with_scaling for each feature set:
#   results_all_raw    = cv_with_scaling(lr_model, df_long, FEATURES_ALL,          'binary_target')
#   results_mi_only    = cv_with_scaling(lr_model, df_long, FEATURES_LONG_MI,       'binary_target')
#   results_fe_only    = cv_with_scaling(lr_model, df_long, FE_FEATURES,            'binary_target')
#   results_final      = cv_with_scaling(lr_model, df_long, FEATURES_FINAL_LONG,    'binary_target')

# TODO: Print all four using print_cv_results()

# TODO: Repeat for SHORT signals (swap df_short, FEATURES_SHORT_MI, FEATURES_FINAL_SHORT)

print('Comparison runs complete ✓')


In [ ]:
# ============================================================
# COMPARISON SUMMARY TABLE
# ============================================================
# Build a clean table so you can see whether engineering helped.

# TODO: Populate this dict with real numbers from the cv runs above
comparison = {
    'Feature Set': [
        'All 33 Raw (FEATURES_ALL)',
        'MI 15-feature (Week 2)',
        'Engineered FE only',
        'Final Selected Set (Week 3)'
    ],
    'n_features': [
        len(FEATURES_ALL),
        len(FEATURES_LONG_MI),
        len(FE_FEATURES),
        # TODO: len(FEATURES_FINAL_LONG)
        None
    ],
    'CV AUC (mean)':     [None, None, None, None],  # TODO: fill in
    'CV AUC (std)':      [None, None, None, None],
    'CV F1 (mean)':      [None, None, None, None],
    'CV Precision (mean)': [None, None, None, None],
    'CV Recall (mean)':  [None, None, None, None],
}

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

# TODO: Comment: Did feature engineering improve AUC? By how much?
# TODO: Comment: Did the final selected set outperform the full raw set?
#                If not, what does that tell you about your feature engineering?


## Section 10 — Per-Symbol Analysis

You trained on all 5 coins pooled. But a crossover on DOGEUSDT has a very different character than one on BTCUSDT. Check whether the pooled model has hidden per-symbol variation — if BTCUSDT AUC is 0.72 but DOGEUSDT is 0.54, you may need symbol-specific models later.


In [ ]:
# ============================================================
# PER-SYMBOL CV EVALUATION (Final Feature Set)
# ============================================================

symbols = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT', 'DOGEUSDT']
sym_results = []

for sym in symbols:
    df_sym = df[(df['signal_x'] == 'LONG') & (df['symbol'] == sym)].sort_values('checked_at_utc')

    if len(df_sym) < 100:
        print(f'{sym}: skipped (fewer than 100 rows)')
        continue

    # TODO: Run cv_with_scaling on df_sym using FEATURES_FINAL_LONG and 'binary_target'
    #       Use n_splits=3 if df_sym is small (fewer than 300 rows)
    #       Use n_splits=5 otherwise

    # TODO: Append to sym_results:
    sym_results.append({
        'symbol':      sym,
        'n_rows':      len(df_sym),
        'pct_positive': round(df_sym['binary_target'].mean() * 100, 1),
        'CV AUC':      None,  # TODO: fill from cv results
        'CV F1':       None,
        'CV Precision': None,
        'CV Recall':   None,
    })

sym_df = pd.DataFrame(sym_results)
print('Per-Symbol Results (LONG, Final Feature Set):')
print(sym_df.to_string(index=False))

# TODO: Comment: Is there a symbol where AUC is significantly lower?
#                What might cause that? (fewer rows? noisier price action? imbalanced classes?)
# TODO: Does the per-symbol variation suggest you should train symbol-specific models?
#       Keep this question open — you'll revisit it when comparing tree models in Week 4.


## Section 11 — Week 3 Honest Baseline Record

Fill this in with real numbers before closing the notebook. This is your updated benchmark. Every model in Weeks 4–10 must beat the AUC and F1 recorded here.


In [ ]:
# ============================================================
# WEEK 3 HONEST BASELINE RECORD
# ============================================================

print("""
=================================================================
         WEEK 3 BASELINE — LogReg + Feature Engineering
=================================================================
  WALK-FORWARD CV FRAMEWORK
    n_splits:                     ___
    Gap:                          ___ (0 if none)
    Scaling:                      Inside each fold (correct pattern)

  FEATURE ENGINEERING SUMMARY
    Original raw features:        33
    Engineered FE_ features:      ___
    Features dropped (corr):      ___
    Features dropped (VIF):       ___
    Features dropped (perm):      ___
    FINAL LONG feature set:       ___ features
    FINAL SHORT feature set:      ___ features

  TOP 5 NEW ENGINEERED FEATURES (by permutation importance)
    1. ___  (importance = ___)
    2. ___
    3. ___
    4. ___
    5. ___

  LOGISTIC REGRESSION — LONG SIGNALS
  (Final feature set, walk-forward CV, 5 folds)
    CV AUC:       ___ ± ___
    CV F1:        ___ ± ___
    CV Precision: ___ ± ___
    CV Recall:    ___ ± ___

  VS WEEK 2 BASELINE
    Week 2 best single-split AUC: ___  (held-out 20%)
    Week 3 CV AUC (mean):         ___
    Improvement from engineering: ___  (can be negative — be honest)

  PER-SYMBOL (LONG, walk-forward CV)
    Best AUC:  ___ on ___
    Worst AUC: ___ on ___
    Largest gap: ___

  LOGISTIC REGRESSION — SHORT SIGNALS
    CV AUC:       ___ ± ___
    CV F1:        ___ ± ___

  HONEST VERDICT
    Is the model better than Week 2?       YES / NO / MARGINAL
    Main source of improvement (if any):   ___
    Main remaining limitation:             ___
    What Week 4 must address:              ___

  DECISION TREE PREVIEW (Week 4)
    Do you expect a tree to capture the boolean/regime features better?
    Write your hypothesis here before seeing the data:
    ___
=================================================================
""")


## Section 12 — Bonus: Recursive Feature Elimination (RFE)

This section is optional for the week but worth attempting if you have time. RFE systematically removes the least important feature, retrains the model, and repeats until the target number of features is reached. It is slower than correlation/VIF/permutation but can find an optimal subset that those filters miss.

Compare the RFE-selected set against your manually filtered set from Section 8. Do they agree? Disagreements reveal which features matter *conditionally* (only in the presence of others).


In [ ]:
# ============================================================
# RECURSIVE FEATURE ELIMINATION (RFE) — BONUS
# ============================================================
# from sklearn.feature_selection import RFE

# TODO: Prepare X (FEATURES_FINAL_LONG, before RFE) and y on LONG signals
# TODO: Use a single 80/20 split for speed (RFE is slow in CV)

# TODO: lr_rfe = LogisticRegression(C=1.0, class_weight='balanced',
#                                    max_iter=1000, solver='lbfgs')
# TODO: rfe = RFE(estimator=lr_rfe, n_features_to_select=20, step=1)
# TODO: rfe.fit(X_train_sc, y_train)

# TODO: Print which features were selected (rfe.support_ == True)
# TODO: Print feature rankings (rfe.ranking_: 1 = selected, higher = dropped earlier)

# TODO: FEATURES_RFE_LONG = [f for f, s in zip(FEATURES_FINAL_LONG, rfe.support_) if s]

# TODO: Evaluate cv_with_scaling using FEATURES_RFE_LONG
#       Compare to your manually-selected FEATURES_FINAL_LONG
#       Which set gives better CV AUC?

# TODO: Do RFE and permutation importance agree on the top 5 features?
#       Where do they disagree? Write a hypothesis for why.

print('RFE complete ✓  (or skipped — mark clearly if skipped)')
